# OSCP LCM-LoRA Training
### Cell Overview
| Cell | Purpose |
|------|----------|
| 1 | Imports |
| 2 | Config dict |
| 3 | Noise schedule + DDIMSolver |
| 4 | Load SD 1.5 components |
| 5 | Inject LoRA into student U-Net |
| 6 | Surrogate classifier + PGD attacker |
| 7 | Dataset + DataLoader |
| 8 | Optimizer + LR scheduler |
| 9 | Training loop |
| 10 | Save LoRA weights |

## Cell 1 — Imports

In [11]:
import os
import math
import random
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Subset

# Diffusion / SD
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler

# CLIP text encoder
from transformers import AutoTokenizer, CLIPTextModel

# LoRA
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict

# Adversarial attack
from advertorch.attacks import LinfPGDAttack

# ---------- Local modules ----------
from ddim_solver import DDIMSolver
from decode_classifier import DecodeClassifier
from archs import get_archs
from dataset import get_dataset

# Math utilities from utils.py (no I/O helpers needed)
from utils import (
    scalings_for_boundary_conditions,
    get_predicted_original_sample,
    get_predicted_noise,
    append_dims,
    encode_prompt,
)

print("Imports OK")

Imports OK


## Cell 2 — Config
All hyperparameters live here.  
Adjust `pretrained_model` to the local SD 1.5 path if HuggingFace Hub is unavailable.

In [12]:
config = dict(
    # Model paths
    pretrained_model   = "stable-diffusion-v1-5/stable-diffusion-v1-5",
    output_dir         = "./logs/OSCP",

    # LoRA parameters
    lora_rank          = 64,
    lora_alpha         = 64,
    lora_dropout       = 0.0,
    # All attention projections + feed-forward + conv layers (from train_lora.py)
    lora_target_modules = [
        "to_q", "to_k", "to_v", "to_out.0",
        "proj_in", "proj_out",
        "ff.net.0.proj", "ff.net.2",
        "conv1", "conv2", "conv_shortcut",
        "downsamplers.0.conv", "upsamplers.0.conv",
        "time_emb_proj",
    ],

    # LCM distillation parameters
    num_ddim_timesteps      = 50,
    timestep_scaling_factor = 10.0,
    loss_type               = "l2",   # "l2" | "huber"
    huber_c                 = 0.001,  # only used when loss_type="huber"

    # N controls the upper bound of the sampled timestep index range:
    # index sampled from [0, num_ddim_timesteps // N)
    N = 2,

    # OSCP adversarial parameters
    surrogate_model      = "resnet50",
    eps                  = 8 / 255,
    attack_iter          = 10,
    eps_iter             = 2 / 255,
    classifier_resolution = 224,
    lmd                  = 0.001,   # weight for latent reconstruction loss term

    # training
    seed                 = 3407,
    train_batch_size     = 4,       # lower for lighter memory use
    max_train_steps      = 5,
    max_train_samples    = 1000,    # set None to use full dataset
    gradient_checkpointing = False, # set True to trade compute for memory

    # optimizer
    learning_rate        = 1e-4,
    adam_beta1           = 0.9,
    adam_beta2           = 0.999,
    adam_weight_decay    = 0.0,
    adam_epsilon         = 1e-8,
    max_grad_norm        = 1.0,
    lr_scheduler         = "constant_with_warmup",
    lr_warmup_steps      = 500,

    # checkpointing
    checkpointing_steps  = 2000,
    vae_encode_batch_size = 8,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(config["seed"])
random.seed(config["seed"])

print(f"Device: {device}")
print(f"Output dir: {config['output_dir']}")

Device: cuda
Output dir: ./logs/OSCP


## Cell 3 — Noise Schedule + DDIMSolver

`DDPMScheduler` provides the $\bar\alpha_t$ (alpha cumprod) schedule.  
`DDIMSolver` subsamples it to `num_ddim_timesteps` evenly-spaced steps used during LCM distillation.

In [13]:
noise_scheduler = DDPMScheduler.from_pretrained(
    config["pretrained_model"], subfolder="scheduler"
)

# alpha_schedule[t] = sqrt(alpha_bar_t)  (the "signal" component)
# sigma_schedule[t] = sqrt(1 - alpha_bar_t)  (the "noise" component)
alpha_schedule = torch.sqrt(noise_scheduler.alphas_cumprod).to(device)
sigma_schedule = torch.sqrt(1 - noise_scheduler.alphas_cumprod).to(device)

# Build the DDIM ODE solver used for the teacher trajectory step
solver = DDIMSolver(
    noise_scheduler.alphas_cumprod.numpy(),
    timesteps=noise_scheduler.config.num_train_timesteps,
    ddim_timesteps=config["num_ddim_timesteps"],
).to(device)

print(f"DDIM timesteps (first 5): {solver.ddim_timesteps[:5].tolist()}")
print(f"Total DDIM steps: {len(solver.ddim_timesteps)}")

DDIM timesteps (first 5): [19, 39, 59, 79, 99]
Total DDIM steps: 50


## Cell 4 — Load SD 1.5 Components

| Component | Trainable | Dtype note |
|-----------|-----------|------------|
| Tokenizer | frozen | — |
| CLIP text encoder | frozen | fp16/bf16 ok |
| VAE | frozen | **keep float32** to avoid NaN losses |
| Teacher U-Net | frozen | fp16 ok (inference only) |
| Student U-Net | **trained** (LoRA only, Cell 5) | float32 required at init |

> **Note:** The teacher U-Net is loaded here but its forward pass is intentionally **not called** during training.  
> Per the OSCP paper, the shifted noise tensor is used as a drop-in teacher output instead (see Cell 9, step 8).

In [ ]:
# Tokenizer  — no gradients, runs on CPU
tokenizer = AutoTokenizer.from_pretrained(
    config["pretrained_model"], subfolder="tokenizer", use_fast=False
)

# CLIP text encoder — frozen
text_encoder = CLIPTextModel.from_pretrained(
    config["pretrained_model"], subfolder="text_encoder"
)
text_encoder.requires_grad_(False)
text_encoder.eval()
text_encoder.to(device)

# VAE — frozen, stays float32 to prevent NaN
vae = AutoencoderKL.from_pretrained(
    config["pretrained_model"], subfolder="vae"
)
vae.requires_grad_(False)
vae.eval()
vae.to(device)   # intentionally no dtype cast

# Teacher U-Net — frozen
# NOTE: its forward pass is replaced by the shifted-noise tensor during training.
# Kept here for reference / to match original code
teacher_unet = UNet2DConditionModel.from_pretrained(
    config["pretrained_model"], subfolder="unet"
)
teacher_unet.requires_grad_(False)
teacher_unet.eval()
teacher_unet.to(device)

# Student U-Net — will be trained via LoRA in Cell 5
unet = UNet2DConditionModel.from_pretrained(
    config["pretrained_model"], subfolder="unet"
)
# Must be float32 at this point (LoRA layers are inserted before any dtype cast)
assert unet.dtype == torch.float32, "Student U-Net must start in float32"
unet.train()
unet.to(device)

print("All SD 1.5 components loaded.")
print(f"  VAE scaling factor : {vae.config.scaling_factor}")
print(f"  Noise pred type    : {noise_scheduler.config.prediction_type}")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 11164.00it/s]
CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All SD 1.5 components loaded.
  VAE scaling factor : 0.18215
  Noise pred type    : epsilon


## Cell 5 — Inject LoRA into Student U-Net

`get_peft_model` wraps all layers matching `lora_target_modules` with a low-rank adapter (rank `r`).  
Only the ~0.5% of LoRA parameters will be updated during training.

In [15]:
lora_config = LoraConfig(
    r=config["lora_rank"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    target_modules=config["lora_target_modules"],
)

unet = get_peft_model(unet, lora_config)

if config["gradient_checkpointing"]:
    # Trades compute for memory: recomputes activations during backward pass
    unet.enable_gradient_checkpointing()

# Shows trainable vs total params — should be ~0.5% trainable
unet.print_trainable_parameters()

trainable params: 67,252,224 || all params: 926,773,188 || trainable%: 7.2566


## Cell 6 — Surrogate Classifier + DecodeClassifier + PGD Attacker

`DecodeClassifier` chains: **latent -> VAE decode -> resize -> classifier**.  
This lets `LinfPGDAttack` compute adversarial gradients in **latent space** rather than pixel space,  
which is the key OSCP design choice (Algorithm 1, step: compute $z^{adv}$).

In [ ]:
# Surrogate classifier (resnet50 with ImageNet pretraining, wrapped with UConn normalization)
classifier = get_archs(config["surrogate_model"], "uconn")
classifier.eval().to(device)

# Bridge: latent space -> pixel space -> classifier logits
decode_classifier = DecodeClassifier(
    classifier=classifier,
    vae=vae,
    scaling_factor=vae.config.scaling_factor,
    size=(config["classifier_resolution"], config["classifier_resolution"]),
).to(device)

# PGD attacker operates entirely in latent space via decode_classifier
adversary = LinfPGDAttack(
    decode_classifier,
    loss_fn=torch.nn.CrossEntropyLoss(reduction="mean"),
    eps=config["eps"],
    nb_iter=config["attack_iter"],
    eps_iter=config["eps_iter"],
    clip_min=None,
    clip_max=None,
    rand_init=False,
    targeted=False,
)

# Quick sanity check: decode_classifier should produce [B, num_classes] logits
with torch.no_grad():
    dummy_latent = torch.randn(2, 4, 28, 28, device=device)  # 224/8 = 28
    dummy_logits = decode_classifier(dummy_latent)
print(f"DecodeClassifier output shape: {dummy_logits.shape}  (expect [2, num_classes])")

DecodeClassifier output shape: torch.Size([2, 2])  (expect [2, num_classes])


## Cell 7 — Dataset + DataLoader

In [17]:
full_dataset = get_dataset("uconn", split="train")

if config["max_train_samples"] is not None:
    n = min(len(full_dataset), config["max_train_samples"])
    indices = random.sample(range(len(full_dataset)), n)
    train_dataset = Subset(full_dataset, indices)
else:
    train_dataset = full_dataset

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config["train_batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

print(f"Dataset size : {len(train_dataset)}")
print(f"Batch size   : {config['train_batch_size']}")
print(f"Batches/epoch: {len(train_dataloader)}")

Dataset size : 1000
Batch size   : 4
Batches/epoch: 250


## Cell 8 — Optimizer + LR Scheduler

In [18]:
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=config["learning_rate"],
    betas=(config["adam_beta1"], config["adam_beta2"]),
    weight_decay=config["adam_weight_decay"],
    eps=config["adam_epsilon"],
)

# Derive epoch count from max_train_steps
num_update_steps_per_epoch = len(train_dataloader)
num_train_epochs = math.ceil(config["max_train_steps"] / num_update_steps_per_epoch)

lr_scheduler = get_scheduler(
    config["lr_scheduler"],
    optimizer=optimizer,
    num_warmup_steps=config["lr_warmup_steps"],
    num_training_steps=config["max_train_steps"],
)

print(f"Epochs to reach {config['max_train_steps']} steps: {num_train_epochs}")
print(f"LR scheduler: {config['lr_scheduler']}")

Epochs to reach 5 steps: 1
LR scheduler: constant_with_warmup


## Cell 9 — Training Loop

Step-by-step translation of the OSCP Algorithm 1:

| Step | Code | Description |
|------|------|-------------|
| 1 | `vae.encode` | $z = \mathcal{E}(x)$ — encode image to latent |
| 2 | `adversary.perturb` | $z^{adv} = $ PGD attack in latent space |
| 3 | `noise = randn + adv - z` | Shifted noise construction (key OSCP trick) |
| 4 | `add_noise` | Forward diffusion: $z_{t_n+k} = \sqrt{\bar\alpha}\,z + \sigma\,\epsilon^{adv}$ |
| 5 | `unet(noisy, t_start)` | Student prediction at start timestep |
| 6 | `c_skip * x + c_out * x0` | LCM boundary condition scaling -> `model_pred` |
| 7 | `ddim_step` | PF-ODE solver step -> `x_prev` at next timestep |
| 8 | `unet(x_prev, t)` (no grad) | Student prediction at target timestep -> `target` |
| 9 | MSE + λ·MSE | LCM loss + latent reconstruction regularisation |

In [ ]:
os.makedirs(config["output_dir"], exist_ok=True)

# pre-compute null (unconditional) prompt embeddings once — shape [1, 77, 768]
with torch.no_grad():
    uncond_input_ids = tokenizer(
        [""],
        return_tensors="pt",
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
    ).input_ids.to(device)
    uncond_prompt_embeds_1 = text_encoder(uncond_input_ids)[0]  # [1, 77, 768]

# stride of the full timestep schedule covered by each DDIM step
topk = noise_scheduler.config.num_train_timesteps // config["num_ddim_timesteps"]

global_step = 0
progress_bar = tqdm(range(config["max_train_steps"]), desc="Steps")

for epoch in range(num_train_epochs):
    for step, (image, label) in enumerate(train_dataloader):

        # 1. Encode images to latents
        image = image.to(device, non_blocking=True)
        label = label.to(device)

        latents_list = []
        with torch.no_grad():
            for i in range(0, image.shape[0], config["vae_encode_batch_size"]):
                chunk = image[i : i + config["vae_encode_batch_size"]]
                # z = E(x)  (Algorithm 1, step 1)
                latents_list.append(vae.encode(chunk).latent_dist.sample())
        latents = torch.cat(latents_list, dim=0) * vae.config.scaling_factor
        bsz = latents.shape[0]

        # 2. Adversarial latent attack: z_adv = PGD(z, y)
        # decode_classifier routes gradients: latent -> VAE decode -> classifier
        adv_latents = adversary.perturb(latents, label)

        # 3. Build shifted noise (OSCP key trick)
        # ε_adv = ε_random + (z_adv − z)  shifts the noise distribution toward z_adv
        noise = torch.randn_like(latents) + adv_latents - latents

        # 4. Sample DDIM timestep index and add noise (forward diffusion)
        # index drawn from [0, num_ddim_timesteps // N) per train_lora.py
        index = torch.randint(
            0, config["num_ddim_timesteps"] // config["N"],
            (bsz,), device=device
        ).long()
        start_timesteps = solver.ddim_timesteps[index]            # t_{n+k}
        timesteps       = start_timesteps - topk                  # t_n
        timesteps       = torch.where(timesteps < 0, torch.zeros_like(timesteps), timesteps)

        # z_{t_{n+k}} = sqrt(alpha_bar) * z + sigma * noise
        noisy_model_input = noise_scheduler.add_noise(latents, noise, start_timesteps)

        # 5. LCM boundary condition scalings
        c_skip_start, c_out_start = scalings_for_boundary_conditions(
            start_timesteps, timestep_scaling=config["timestep_scaling_factor"]
        )
        c_skip_start = append_dims(c_skip_start, latents.ndim)
        c_out_start  = append_dims(c_out_start,  latents.ndim)

        c_skip, c_out = scalings_for_boundary_conditions(
            timesteps, timestep_scaling=config["timestep_scaling_factor"]
        )
        c_skip = append_dims(c_skip, latents.ndim)
        c_out  = append_dims(c_out,  latents.ndim)

        # Broadcast unconditional embeds to batch size
        prompt_embeds = uncond_prompt_embeds_1.expand(bsz, -1, -1)

        # 6. Student U-Net forward at start timestep
        noise_pred = unet(
            noisy_model_input,
            start_timesteps,
            timestep_cond=None,
            encoder_hidden_states=prompt_embeds,
        ).sample

        # Recover x_0 prediction, then apply LCM skip/out boundary scaling
        pred_x_0 = get_predicted_original_sample(
            noise_pred, start_timesteps, noisy_model_input,
            noise_scheduler.config.prediction_type,
            alpha_schedule, sigma_schedule,
        )
        model_pred = c_skip_start * noisy_model_input + c_out_start * pred_x_0

        # 7. Teacher trajectory step (no grad)
        # NOTE: The teacher U-Net forward pass is intentionally skipped.
        # The shifted noise tensor IS the teacher output in OSCP.
        # This is the paper's key substitution: ε_teacher ← ε_adv
        with torch.no_grad():
            uncond_teacher_output = noise   # ← OSCP: shifted noise replaces teacher

            pred_x0 = get_predicted_original_sample(
                uncond_teacher_output, start_timesteps, noisy_model_input,
                noise_scheduler.config.prediction_type,
                alpha_schedule, sigma_schedule,
            )
            pred_noise_teacher = get_predicted_noise(
                uncond_teacher_output, start_timesteps, noisy_model_input,
                noise_scheduler.config.prediction_type,
                alpha_schedule, sigma_schedule,
            )
            # PF-ODE step: x_{t_n} = sqrt(alpha_bar_{t_n}) * x_0 + sigma_{t_n} * eps
            x_prev = solver.ddim_step(pred_x0, pred_noise_teacher, index)

        # 8. Student forward at target timestep (consistency target)
        with torch.no_grad():
            target_noise_pred = unet(
                x_prev.float(),
                timesteps,
                timestep_cond=None,
                encoder_hidden_states=prompt_embeds,
            ).sample
            pred_x_0_target = get_predicted_original_sample(
                target_noise_pred, timesteps, x_prev,
                noise_scheduler.config.prediction_type,
                alpha_schedule, sigma_schedule,
            )
            target = c_skip * x_prev + c_out * pred_x_0_target

        # 9. Loss
        # LCM consistency loss + latent reconstruction regularisation (weighted by lambda)
        if config["loss_type"] == "l2":
            loss = (
                F.mse_loss(model_pred.float(), target.float(), reduction="mean")
                + config["lmd"] * F.mse_loss(model_pred.float(), latents.float(), reduction="mean")
            )
        elif config["loss_type"] == "huber":
            c = config["huber_c"]
            loss = (
                torch.mean(torch.sqrt((model_pred.float() - target.float()) ** 2 + c**2) - c)
                + config["lmd"] * torch.mean(
                    torch.sqrt((model_pred.float() - latents.float()) ** 2 + c**2) - c
                )
            )

        # 10. Backward + optimizer step
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), config["max_grad_norm"])
        optimizer.step()
        lr_scheduler.step()

        # Logging + checkpointing
        global_step += 1
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr_scheduler.get_last_lr()[0]:.2e}")

        if global_step % config["checkpointing_steps"] == 0:
            ckpt_path = os.path.join(config["output_dir"], f"checkpoint-{global_step}.pt")
            torch.save(
                {
                    "global_step": global_step,
                    "unet_lora_state_dict": get_peft_model_state_dict(unet),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "lr_scheduler_state_dict": lr_scheduler.state_dict(),
                },
                ckpt_path,
            )
            print(f"\nCheckpoint saved -> {ckpt_path}")

        if global_step >= config["max_train_steps"]:
            break

    if global_step >= config["max_train_steps"]:
        break

print(f"\nTraining complete. Total steps: {global_step}")

Steps: 100%|██████████| 1/1 [18:53<00:00, 1133.83s/it, loss=0.0041, lr=2.00e-07]



Training complete. Total steps: 5


## Cell 10 — Save Final LoRA Weights

In [ ]:
lora_save_dir = os.path.join(config["output_dir"], "unet_lora")
os.makedirs(lora_save_dir, exist_ok=True)

# delete existing file before overwrite
existing = os.path.join(lora_save_dir, "pytorch_lora_weights.safetensors")
if os.path.exists(existing):
    os.remove(existing)


lora_state_dict = get_peft_model_state_dict(unet, adapter_name="default")
StableDiffusionPipeline.save_lora_weights(lora_save_dir, lora_state_dict)

print(f"LoRA weights saved at: {lora_save_dir}")
print("Files written:")
for f in os.listdir(lora_save_dir):
    print(f"  {f}")

LoRA weights saved → ./logs/OSCP\unet_lora
Files written:
  pytorch_lora_weights.safetensors
